# Disaster Tweet Classification: Pretrained DeBERTa-v3 (deberta-v3-base) Fine-Tuning

Includes Dataset EDA, Pretrained Tokenization, Mixed-Precision Fine-Tuning with Training/Val Curves, Dual Confusion Matrices, and Per-Class Metric plots.


In [ ]:
import os
import re
import html
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid', palette='muted')

# Auto-detect data path (Local workspace vs Kaggle environment)
DATA_DIR = Path("dataset")
if not DATA_DIR.exists():
    kaggle_paths = list(Path("/kaggle/input").glob("**/train.parquet"))
    if kaggle_paths:
        DATA_DIR = kaggle_paths[0].parent
    else:
        DATA_DIR = Path("dataset_csv")

print(f"[+] Active dataset directory: {DATA_DIR}")

if (DATA_DIR / "train.parquet").exists():
    train_df = pd.read_parquet(DATA_DIR / "train.parquet")
    val_df = pd.read_parquet(DATA_DIR / "validation.parquet")
    test_df = pd.read_parquet(DATA_DIR / "test.parquet")
else:
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    val_df = pd.read_csv(DATA_DIR / "validation.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train Split : {train_df.shape} ({len(train_df):,} samples)")
print(f"Val Split   : {val_df.shape} ({len(val_df):,} samples)")
print(f"Test Split  : {test_df.shape} ({len(test_df):,} samples)")


In [ ]:
# --------------------------------------------------------------------------
# 📊 Dataset Insights & EDA Visualizations
# --------------------------------------------------------------------------
def plot_dataset_insights(train_df, val_df, test_df):
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # 1. Class Distribution Across Splits
    classes = sorted(train_df['class_label'].unique())
    dist_df = pd.DataFrame(index=classes)
    dist_df['Train'] = train_df['class_label'].value_counts()
    dist_df['Val'] = val_df['class_label'].value_counts()
    dist_df['Test'] = test_df['class_label'].value_counts()
    dist_df = dist_df.sort_values(by='Train', ascending=True)
    
    clean_labels = [c.replace('_', ' ').title() for c in dist_df.index]
    dist_df.index = clean_labels
    dist_df.plot(kind='barh', stacked=True, ax=axes[0], color=['#2b5c8f', '#e67e22', '#27ae60'], edgecolor='black', alpha=0.85)
    axes[0].set_title('Class Distribution Across Splits (HumAID 10 Classes)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Number of Tweets')
    axes[0].legend(title='Split', frameon=True)
    
    # 2. Tweet Length Distribution
    for name, df, col in [('Train', train_df, '#2b5c8f'), ('Val', val_df, '#e67e22'), ('Test', test_df, '#27ae60')]:
        word_lens = df['tweet_text'].apply(lambda x: len(str(x).split()))
        sns.kdeplot(word_lens, ax=axes[1], label=f"{name} (mean={word_lens.mean():.1f} words)", color=col, fill=True, alpha=0.25)
        
    axes[1].set_title('Tweet Word Count Distribution (KDE)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Word Count per Tweet')
    axes[1].set_ylabel('Density')
    axes[1].set_xlim(0, 50)
    axes[1].legend(title='Split', frameon=True)
    
    plt.tight_layout()
    plt.show()

plot_dataset_insights(train_df, val_df, test_df)


In [ ]:
# --------------------------------------------------------------------------
# Preprocessing Cleaners (Standard, Light, Raw, Aggressive)
# --------------------------------------------------------------------------
CONTRACTIONS = {
    "can't": "cannot", "won't": "will not", "n't": " not", "'re": " are",
    "'s": " is", "'d": " would", "'ll": " will", "'t": " not", "'ve": " have",
    "'m": " am", "i'm": "i am", "we're": "we are", "they're": "they are",
    "it's": "it is", "there's": "there is", "that's": "that is", "what's": "what is"
}

DISASTER_SLANG = {
    r"\bpls\b": "please", r"\bplz\b": "please", r"\bthx\b": "thanks",
    r"\bu\b": "you", r"\bur\b": "your", r"\br\b": "are",
    r"\bw/\b": "with", r"\bw/o\b": "without", r"\bb4\b": "before",
    r"\bmsg\b": "message", r"\binfo\b": "information", r"\bemerg\b": "emergency",
    r"\bevac\b": "evacuation", r"\bevacs\b": "evacuations", r"\bvicts\b": "victims",
    r"\bgov\b": "government", r"\bdept\b": "department", r"\bvol\b": "volunteer"
}

EMOJI_TRANSLATIONS = {
    "🙏": " prayer support ", "💔": " heartbreak grief ", "❤️": " love sympathy ",
    "🚨": " emergency warning alert ", "⚠️": " danger warning caution ",
    "🔥": " fire wildfire disaster ", "🌊": " flood tsunami water surge ",
    "🌧️": " rain storm hurricane ", "🌪️": " tornado storm ", "⚡": " storm lightning ",
    "😢": " crying sorrow sadness ", "😭": " weeping tragedy ", "🕯️": " mourning memorial ",
    "🆘": " urgent help request emergency ", "🏠": " shelter home house ", "🏥": " hospital medical clinic "
}

def split_camel_case(text: str) -> str:
    return re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

def clean_raw(text: str) -> str:
    return str(text) if text is not None else ""

def clean_light(text: str) -> str:
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r'https?://\S+|www\.\S+', '[URL]', text)
    text = re.sub(r'@\w+', '[USER]', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_standard(text: str) -> str:
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    for em, rep in EMOJI_TRANSLATIONS.items(): text = text.replace(em, rep)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#(\w+)', lambda m: split_camel_case(m.group(1)), text)
    text = text.lower()
    for c, exp in CONTRACTIONS.items(): text = text.replace(c, exp)
    for p, rep in DISASTER_SLANG.items(): text = re.sub(p, rep, text, flags=re.IGNORECASE)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\brt\b', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_aggressive(text: str) -> str:
    import string
    text = clean_standard(text)
    text = text.translate(str.maketrans('', '', string.punctuation + string.digits))
    stopwords = {'the', 'a', 'an', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'it', 'this', 'that', 'from', 'as', 'be', 'have', 'has', 'had', 'do', 'does', 'did', 'but', 'not', 'so', 'we', 'i', 'you', 'they', 'he', 'she', 'my', 'your', 'our', 'their'}
    words = [w for w in text.split() if w not in stopwords and len(w) > 2]
    return ' '.join(words)


In [ ]:
# --------------------------------------------------------------------------
# 📈 Comprehensive Evaluation Suite & Metric Plotting Functions
# --------------------------------------------------------------------------
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

def plot_comprehensive_evaluation(y_true, y_pred, class_names, model_title="Model", y_train=None):
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    w_p, w_r, w_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    p_per, r_per, f1_per, support = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    
    clean_classes = [c.replace('_', ' ').title() for c in class_names]
    
    # Print Terminal Report
    print("=" * 75)
    print(f"               {model_title.upper()} — EVALUATION SUMMARY")
    print("=" * 75)
    print(f"Accuracy          : {acc * 100:.2f}%")
    print(f"Macro F1-Score    : {macro_f1 * 100:.2f}%  <-- [Primary Optimization Metric]")
    print(f"Weighted F1-Score : {w_f1 * 100:.2f}%")
    print(f"Macro Precision   : {macro_p * 100:.2f}%")
    print(f"Macro Recall      : {macro_r * 100:.2f}%")
    print("=" * 75)
    print("\nDetailed Per-Class Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=clean_classes, digits=4, zero_division=0))
    
    # 1. Dual Confusion Matrices (Normalized & Raw Counts)
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    cm_norm = confusion_matrix(y_true, y_pred, normalize='true')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[0], xticklabels=clean_classes, yticklabels=clean_classes)
    axes[0].set_title(f"{model_title} — Normalized Confusion Matrix (Macro-F1: {macro_f1*100:.2f}%)", fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Predicted Class', fontsize=11)
    axes[0].set_ylabel('True Class', fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)
    
    cm_raw = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Oranges', ax=axes[1], xticklabels=clean_classes, yticklabels=clean_classes)
    axes[1].set_title(f"{model_title} — Raw Counts Confusion Matrix", fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Predicted Class', fontsize=11)
    axes[1].set_ylabel('True Class', fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # 2. Per-Class Triple Bar Chart (Precision, Recall, F1)
    df_metrics = pd.DataFrame({
        'Class': clean_classes,
        'Precision (%)': p_per * 100,
        'Recall (%)': r_per * 100,
        'F1-Score (%)': f1_per * 100
    }).sort_values(by='F1-Score (%)', ascending=True)
    
    fig, ax = plt.subplots(figsize=(14, 7))
    df_metrics.plot(x='Class', y=['Precision (%)', 'Recall (%)', 'F1-Score (%)'], kind='barh', ax=ax, color=['#3498db', '#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.9)
    ax.set_title(f"{model_title} — Per-Class Precision, Recall & F1-Score Breakdown", fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Score (%)', fontsize=12)
    ax.set_xlim(0, 105)
    ax.legend(frameon=True, fontsize=11)
    plt.tight_layout()
    plt.show()
    
    # 3. Class Imbalance vs F1-Score Correlation (if y_train is provided)
    if y_train is not None:
        train_counts = pd.Series(y_train).value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.regplot(x=train_counts.values, y=f1_per * 100, ax=ax, scatter_kws={'s': 100, 'color': '#2c3e50'}, line_kws={'color': '#e74c3c', 'linestyle': '--'})
        ax.set_title('Training Sample Count vs Test Per-Class F1-Score (Imbalance Robustness)', fontsize=13, fontweight='bold')
        ax.set_xlabel('Number of Training Samples (Class Frequency)', fontsize=11)
        ax.set_ylabel('Test F1-Score (%)', fontsize=11)
        for i, txt in enumerate(clean_classes):
            ax.annotate(txt, (train_counts.values[i], f1_per[i] * 100 + 1.0), fontsize=9)
        plt.tight_layout()
        plt.show()


In [ ]:
# --------------------------------------------------------------------------
# 📉 Training Loss & Validation Macro-F1 Curves Plotting
# --------------------------------------------------------------------------
def plot_training_history(train_losses, val_f1s, model_name="Neural Model"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(train_losses) + 1)
    
    # Train Loss
    ax1.plot(epochs, train_losses, 'o-', color='#e74c3c', linewidth=2, label='Training Loss')
    ax1.set_title(f'{model_name} — Training Loss per Epoch', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Cross-Entropy Loss')
    ax1.legend(frameon=True)
    ax1.grid(True, linestyle='--', alpha=0.6)
    
    # Val Macro-F1
    ax2.plot(epochs, [f * 100 for f in val_f1s], 's-', color='#27ae60', linewidth=2, label='Val Macro-F1 (%)')
    ax2.set_title(f'{model_name} — Validation Macro-F1 (%) Progression', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Macro F1 (%)')
    ax2.legend(frameon=True)
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()


## 1. Apply Light Preprocessing (Preserve Casing & Punctuation)


In [ ]:
train_df['clean_text'] = train_df['tweet_text'].apply(clean_light)
val_df['clean_text'] = val_df['tweet_text'].apply(clean_light)
test_df['clean_text'] = test_df['tweet_text'].apply(clean_light)

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['class_label'])
y_val = label_encoder.transform(val_df['class_label'])
y_test = label_encoder.transform(test_df['class_label'])
classes = list(label_encoder.classes_)
num_classes = len(classes)

class_weights = compute_class_weight('balanced', classes=np.arange(num_classes), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print("Target Classes:", classes)


## 2. Load `microsoft/deberta-v3-base` Tokenizer & Build DataLoaders


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader

MODEL_CHECKPOINT = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

class TransformerTweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=96):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

print(f"[+] Tokenizing splits with {MODEL_CHECKPOINT}...")
train_dataset = TransformerTweetDataset(train_df['clean_text'].tolist(), y_train, tokenizer)
val_dataset = TransformerTweetDataset(val_df['clean_text'].tolist(), y_val, tokenizer)
test_dataset = TransformerTweetDataset(test_df['clean_text'].tolist(), y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


## 3. Initialize Model with Automatic Mixed Precision (AMP)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Active computation device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=num_classes).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


## 4. Fine-Tuning Loop with Validation Monitoring


In [ ]:
EPOCHS = 3
best_val_f1 = 0.0
best_model_state = None
train_losses, val_f1_history = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Validation
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(batch['labels'].numpy())
            
    _, _, val_f1, _ = precision_recall_fscore_support(val_targets, val_preds, average='macro', zero_division=0)
    val_f1_history.append(val_f1)
    print(f"Epoch {epoch:02d} | Train Loss: {avg_loss:.4f} | Val Macro-F1: {val_f1*100:.2f}%")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict().copy()

if best_model_state: model.load_state_dict(best_model_state)

plot_training_history(train_losses, val_f1_history, model_name="DeBERTa-v3 (deberta-v3-base)")


## 5. Comprehensive Test Split Evaluation Suite & Plots


In [ ]:
model.eval()
test_preds, test_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_targets.extend(batch['labels'].numpy())

plot_comprehensive_evaluation(np.array(test_targets), np.array(test_preds), classes, model_title="DeBERTa-v3 (deberta-v3-base)", y_train=y_train)
